# HGT4Rec — standardized preprocessing pipeline

One notebook, parameterized by Mongo DB names. Takes a partial Goodreads slice
(books / reviews / interactions in a slice DB; authors / works / genres in a meta
DB) and produces every artifact in `data/processed/` needed for training, plus
the temporal splits and the assembled `hetero_data.pt`.

Neo4j is **not** an intermediate — it's an optional export at the very end for
visualization. To point this at a new partial slice, change `SLICE_DB` in the
config cell below and run all cells.

**Pipeline stages** (one per section):

1. Setup & config
2. Helpers
3. Load raw data from Mongo
4. Normalize categorical fields
5. Build `*_to_idx.json` (all 10 node types)
6. Text embeddings (book / review / author / work, all 4 in one cell)
7. Numeric blocks (one cell per node type)
8. Derive edges from Mongo (replaces every Neo4j cypher)
9. Temporal splits → `splits/{train,val,test}.npz`
10. Assemble `HeteroData` → `hetero_data.pt`
11. Sanity check
12. **(Optional)** Neo4j export for visualization

## 1. Setup & config

Edit values here to point at a different slice.

In [ ]:
from __future__ import annotations

import gc
import gzip
import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Any

import h5py
import numpy as np
import pandas as pd
import torch
from pymongo import MongoClient
from tqdm.auto import tqdm

# ── Slice configuration ───────────────────────────────────────────────────────
MONGO_URI    = "mongodb://localhost:27017/"
SLICE_DB     = "poetry"             # change to point at another partial slice
META_DB      = "goodreads"          # holds authors / works / genres lookups
SLICE_NAME   = "poetry"             # used in run banner and (optional) Neo4j db name
OUT_DIR      = Path("data/processed")
SPLITS_DIR   = OUT_DIR / "splits"
SEED         = 42

# ── Embedding configuration ───────────────────────────────────────────────────
EMB_MODEL    = "BAAI/bge-base-en-v1.5"
EMB_DIM      = 768
EMB_BATCH    = 256                   # lower to 128 if GPU OOM
MAX_CHARS    = 2000                  # ~512 tokens — truncation for safety

# ── Splits ────────────────────────────────────────────────────────────────────
TRAIN_YEARS_BEFORE_T = 2             # train: date_added < T-2yr
VAL_YEARS_BEFORE_T   = 1             # val:   [T-2yr, T-1yr); test: >= T-1yr

# ── Categorical bucket thresholds (UNK / OTHER tail bucketing) ───────────────
LANG_MIN_COUNT      = 50
FORMAT_MIN_COUNT    = 50
PUBLISHER_MIN_COUNT = 50
SHELF_TOP_N         = 60

# Drop HAS_GENRE edges below this normalized-weight threshold (per book, fractions sum to 1).
# 0.25 means: keep only genres that account for ≥25% of a book's classification weight.
# Set to 0.0 to disable the filter (keep all genre edges).
GENRE_MIN_WEIGHT    = 0.25

SHELF_STOPLIST = {
    "to-read", "currently-reading", "read", "owned", "own", "books-i-own",
    "favorites", "my-books", "default", "kindle", "ebook", "ebooks",
    "audiobook", "audio", "library", "wishlist", "want-to-read",
    "did-not-finish", "dnf", "abandoned", "general", "shelfari-favorites",
    "all", "maybe", "re-read", "new", "recommended", "owned-books",
    "to-buy", "my-library", "i-own", "required-reading", "home-library",
    "my-ebooks", "to-re-read",
}

# ── Optional Neo4j export at the end ─────────────────────────────────────────
PUSH_TO_NEO4J = False
NEO4J_URI     = "bolt://localhost:7687"
NEO4J_AUTH    = ("neo4j", "Niveau99")
NEO4J_DB      = SLICE_NAME            # writes to a database matching SLICE_NAME

# ── Boilerplate ──────────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Slice DB    : {SLICE_DB}")
print(f"Meta DB     : {META_DB}")
print(f"Output dir  : {OUT_DIR.resolve()}")
print(f"Device      : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Helpers

Normalization rules + small utilities. All slice-independent.

In [ ]:
def normalize_language(code) -> str:
    if pd.isna(code) or code in ("", "--"):
        return "UNK"
    code = code.lower().strip()
    if code in {"eng", "en", "en-us", "en-ca", "en-gb"}: return "eng"
    if code in {"ger", "deu"}:                            return "ger"
    if code in {"fre", "fra", "fr"}:                      return "fre"
    if code in {"nl",  "dut", "nld"}:                     return "dut"
    if code in {"gre", "ell"}:                            return "gre"
    if code in {"per", "fas", "pes"}:                     return "per"
    if code in {"nor", "nob", "nno"}:                     return "nor"
    return code


def normalize_format(fmt) -> str:
    if pd.isna(fmt) or fmt == "":
        return "UNK"
    return fmt.strip().title()


def normalize_publisher(name) -> str:
    if pd.isna(name) or name == "":
        return "UNK"
    return name.strip()


def bucket_by_min_count(series: pd.Series, min_count: int, other_label: str = "OTHER") -> pd.Series:
    """Replace values with count < min_count by `other_label`. Preserves UNK."""
    counts = series.value_counts()
    keep   = set(counts[counts >= min_count].index)
    return series.where(series.isin(keep), other_label)


def parse_gr_date(s):
    """Goodreads date format: 'Wed Mar 14 12:35:00 -0700 2018' → tz-aware Timestamp."""
    if not s or (isinstance(s, float) and np.isnan(s)):
        return None
    try:
        return pd.to_datetime(s, format="%a %b %d %H:%M:%S %z %Y")
    except (ValueError, TypeError):
        return None


def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def save_json(obj, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    print(f"  saved {path}  ({len(obj) if hasattr(obj, '__len__') else '?'} entries)")

## 3. Load raw data from Mongo

Five DataFrames:

| Variable      | Source collection                  | Filter |
|---             |---                                 |---|
| `books_df`    | `{SLICE_DB}.books`                | all books in the slice |
| `reviews_df`  | `{SLICE_DB}.reviews`              | filtered to slice books, drop empty `review_text` |
| `inters_df`   | `{SLICE_DB}.interactions`         | filtered to slice books |
| `authors_df`  | `{META_DB}.authors`               | restricted to author_ids appearing in `books_df.authors[]` |
| `works_df`    | `{META_DB}.works`                 | restricted to work_ids appearing in `books_df.work_id` |
| `genres_df`   | `{META_DB}.genres`                | restricted to slice book_ids |

In [ ]:
client    = MongoClient(MONGO_URI)
slice_db  = client[SLICE_DB]
meta_db   = client[META_DB]

print(f"\n── Loading {SLICE_DB}.books ──")
books_df = pd.DataFrame(slice_db.books.find({}, {"_id": 0}))
books_df["book_id"] = books_df["book_id"].astype(str)
print(f"  {len(books_df):,} books")

book_ids_list = books_df["book_id"].unique().tolist()
BATCH = 5_000

# ── Reviews — only for books in this slice ───────────────────────────────────
print(f"\n── Loading {SLICE_DB}.reviews ──")
review_chunks = []
for chunk in tqdm(list(chunked(book_ids_list, BATCH)), desc="reviews"):
    cursor = slice_db.reviews.find({"book_id": {"$in": chunk}}, {"_id": 0})
    review_chunks.append(pd.DataFrame(list(cursor)))
reviews_df = pd.concat(review_chunks, ignore_index=True) if review_chunks else pd.DataFrame()
print(f"  {len(reviews_df):,} reviews")

# Drop empty review_text (canonical: only reviews with text become Review nodes)
reviews_df["review_text"] = reviews_df["review_text"].replace("", None)
reviews_df = reviews_df[reviews_df["review_text"].notna()].reset_index(drop=True)
reviews_df["review_id"] = reviews_df["review_id"].astype(str)
reviews_df["book_id"]   = reviews_df["book_id"].astype(str)
reviews_df["user_id"]   = reviews_df["user_id"].astype(str)
for col in ["rating", "n_votes", "n_comments"]:
    reviews_df[col] = pd.to_numeric(reviews_df.get(col), errors="coerce")
for col in ["date_added", "date_updated", "read_at", "started_at"]:
    if col in reviews_df.columns:
        reviews_df[col] = reviews_df[col].apply(parse_gr_date)
print(f"  {len(reviews_df):,} reviews with text")

# ── Interactions ──────────────────────────────────────────────────────────────
print(f"\n── Loading {SLICE_DB}.interactions ──")
inters_chunks = []
for chunk in tqdm(list(chunked(book_ids_list, BATCH)), desc="interactions"):
    cursor = slice_db.interactions.find(
        {"book_id": {"$in": chunk}},
        {"_id": 0, "user_id": 1, "book_id": 1, "review_id": 1,
         "is_read": 1, "rating": 1, "date_added": 1},
    )
    inters_chunks.append(pd.DataFrame(list(cursor)))
inters_df = pd.concat(inters_chunks, ignore_index=True) if inters_chunks else pd.DataFrame()
print(f"  {len(inters_df):,} interactions")

inters_df["rating"]  = pd.to_numeric(inters_df["rating"], errors="coerce").fillna(0).astype(int)
inters_df["is_read"] = inters_df["is_read"].astype(bool)
inters_df["user_id"] = inters_df["user_id"].astype(str)
inters_df["book_id"] = inters_df["book_id"].astype(str)
inters_df["date_added"] = inters_df["date_added"].apply(parse_gr_date)

# ── Authors ──────────────────────────────────────────────────────────────────
print(f"\n── Loading {META_DB}.authors ──")
author_ids_in_slice = sorted({
    a["author_id"]
    for authors in books_df["authors"]
    for a in (authors or [])
})
authors_df = pd.DataFrame(list(meta_db.authors.find(
    {"author_id": {"$in": author_ids_in_slice}}, {"_id": 0}
)))
authors_df["author_id"] = authors_df["author_id"].astype(str)
for col in ["average_rating", "ratings_count", "text_reviews_count"]:
    if col in authors_df.columns:
        authors_df[col] = pd.to_numeric(authors_df[col], errors="coerce")
print(f"  needed {len(author_ids_in_slice):,} authors, fetched {len(authors_df):,}")

# ── Works ────────────────────────────────────────────────────────────────────
print(f"\n── Loading {META_DB}.works ──")
work_ids_in_slice = books_df["work_id"].dropna().astype(str).unique().tolist()
works_df = pd.DataFrame(list(meta_db.works.find(
    {"work_id": {"$in": work_ids_in_slice}}, {"_id": 0}
)))
works_df["work_id"] = works_df["work_id"].astype(str)
print(f"  needed {len(work_ids_in_slice):,} works, fetched {len(works_df):,}")

# Parse work rating distribution string
def parse_rd(s):
    if not s or (isinstance(s, float) and np.isnan(s)):
        return {}
    return {k: int(v) for k, v in (p.split(":") for p in s.split("|"))}

if "rating_dist" in works_df.columns:
    works_df["rd_parsed"] = works_df["rating_dist"].apply(parse_rd)
    for k in ["1", "2", "3", "4", "5", "total"]:
        works_df[f"rd_{k}"] = works_df["rd_parsed"].apply(lambda d: d.get(k))
else:
    for k in ["1", "2", "3", "4", "5", "total"]:
        works_df[f"rd_{k}"] = None

for col in ["books_count", "ratings_count", "text_reviews_count",
            "original_publication_year",
            "rd_1", "rd_2", "rd_3", "rd_4", "rd_5", "rd_total"]:
    if col in works_df.columns:
        works_df[col] = pd.to_numeric(works_df[col], errors="coerce")

# ── Genres ───────────────────────────────────────────────────────────────────
print(f"\n── Loading {META_DB}.genres ──")
genres_df = pd.DataFrame(list(meta_db.genres.find(
    {"book_id": {"$in": book_ids_list}}, {"_id": 0, "book_id": 1, "genres": 1}
)))
if "book_id" in genres_df.columns:
    genres_df["book_id"] = genres_df["book_id"].astype(str)
print(f"  {len(genres_df):,} books with genre annotations")

# ── Sanity assertions ────────────────────────────────────────────────────────
assert books_df["book_id"].is_unique,   "duplicate book_id in slice"
assert reviews_df["review_id"].is_unique, "duplicate review_id in slice"

## 4. Normalize categorical fields

Apply the language/format/publisher normalizers + tail-bucket to OTHER.

In [ ]:
books_df["language_norm"]  = books_df["language_code"].apply(normalize_language)
books_df["language_node"]  = bucket_by_min_count(books_df["language_norm"], LANG_MIN_COUNT)

books_df["format_norm"]    = books_df["format"].apply(normalize_format)
books_df["format_node"]    = bucket_by_min_count(books_df["format_norm"], FORMAT_MIN_COUNT)

books_df["publisher_norm"] = books_df["publisher"].apply(normalize_publisher)
books_df["publisher_node"] = bucket_by_min_count(books_df["publisher_norm"], PUBLISHER_MIN_COUNT)

print("Language distribution:")
print(books_df["language_node"].value_counts().head(10).to_string())
print("\nFormat distribution:")
print(books_df["format_node"].value_counts().head(10).to_string())
print(f"\nPublisher: {books_df['publisher_node'].nunique()} distinct nodes")

## 5. Build `*_to_idx.json`

Canonical ordering across the pipeline: **lexicographic sort of the string ID**
(this is what `sorted(list_of_strings)` does in Python). Every HDF5 row order
and every edge_index downstream is derived from these maps, so getting the
order right here is load-bearing.

In [ ]:
# ── book_id_to_idx ───────────────────────────────────────────────────────────
book_ids_sorted = sorted(books_df["book_id"].astype(str).unique())
book_to_idx     = {bid: i for i, bid in enumerate(book_ids_sorted)}
save_json(book_to_idx, OUT_DIR / "book_id_to_idx.json")

# ── work_to_idx ──────────────────────────────────────────────────────────────
work_ids_sorted = sorted(works_df["work_id"].astype(str).unique())
work_to_idx     = {wid: i for i, wid in enumerate(work_ids_sorted)}
save_json(work_to_idx, OUT_DIR / "work_to_idx.json")

# ── author_to_idx (only authors actually present in goodreads.authors) ───────
author_ids_sorted = sorted(authors_df["author_id"].astype(str).unique())
author_to_idx     = {aid: i for i, aid in enumerate(author_ids_sorted)}
save_json(author_to_idx, OUT_DIR / "author_to_idx.json")

# ── review_to_idx ────────────────────────────────────────────────────────────
review_ids_sorted = sorted(reviews_df["review_id"].astype(str).unique())
review_to_idx     = {rid: i for i, rid in enumerate(review_ids_sorted)}
save_json(review_to_idx, OUT_DIR / "review_to_idx.json")

# ── user_to_idx (union of reviews + interactions user_ids) ───────────────────
user_ids_sorted = sorted(set(reviews_df["user_id"].astype(str)) |
                         set(inters_df["user_id"].astype(str)))
user_to_idx     = {uid: i for i, uid in enumerate(user_ids_sorted)}
save_json(user_to_idx, OUT_DIR / "user_to_idx.json")

# ── language / format / publisher (bucketed) ─────────────────────────────────
def vocab_idx(series: pd.Series) -> dict:
    return {v: i for i, v in enumerate(sorted(series.unique()))}

lang_to_idx = vocab_idx(books_df["language_node"]);  save_json(lang_to_idx, OUT_DIR / "language_to_idx.json")
fmt_to_idx  = vocab_idx(books_df["format_node"]);     save_json(fmt_to_idx,  OUT_DIR / "format_to_idx.json")
pub_to_idx  = vocab_idx(books_df["publisher_node"]);  save_json(pub_to_idx,  OUT_DIR / "publisher_to_idx.json")

# ── genre_to_idx (from goodreads.genres) ─────────────────────────────────────
genre_names = sorted({g for row in genres_df.get("genres", []) for g in (row or {})})
genre_to_idx = {name: i for i, name in enumerate(genre_names)}
save_json(genre_to_idx, OUT_DIR / "genre_to_idx.json")

# ── shelf_to_idx (top-N by total count, stoplist removed) ────────────────────
shelf_counts: dict[str, int] = defaultdict(int)
for shelves in books_df["popular_shelves"]:
    for s in (shelves or []):
        shelf_counts[s["name"]] += int(s["count"])
shelf_df_local = (
    pd.DataFrame(shelf_counts.items(), columns=["shelf", "total_count"])
    .query("shelf not in @SHELF_STOPLIST")
    .sort_values("total_count", ascending=False)
    .head(SHELF_TOP_N)
)
shelf_vocab = sorted(shelf_df_local["shelf"].tolist())
shelf_to_idx = {s: i for i, s in enumerate(shelf_vocab)}
save_json(shelf_to_idx, OUT_DIR / "shelf_to_idx.json")

## 6. Text embeddings

One HDF5 per node type with text. Row order = `sorted(id)` ascending lex —
identical to the idx maps. All four use the same BGE model loaded once.

In [ ]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {EMB_MODEL} on {device}...")
emb_model = SentenceTransformer(EMB_MODEL, device=device)


def embed_and_save(ids_sorted: list[str], texts_by_id: dict[str, str],
                   id_key: str, h5_path: Path,
                   *, max_chars: int = MAX_CHARS, fallback: str = "[no text]"):
    texts = []
    for k in ids_sorted:
        t = (texts_by_id.get(k) or "").strip()
        texts.append((t or fallback)[:max_chars])

    print(f"\n── {h5_path.name}: {len(texts):,} texts")
    embeddings = emb_model.encode(
        texts,
        batch_size=EMB_BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    assert embeddings.shape == (len(ids_sorted), EMB_DIM), embeddings.shape

    with h5py.File(h5_path, "w") as hf:
        hf.create_dataset(id_key,    data=np.array(ids_sorted, dtype="S40"))
        hf.create_dataset("embedding", data=embeddings.astype(np.float32),
                          compression="gzip", compression_opts=4)
    print(f"  saved {h5_path}  shape={embeddings.shape}  norm[0:3]="
          f"{np.linalg.norm(embeddings[:3], axis=1).round(4)}")


# ── Books: title + description ───────────────────────────────────────────────
book_texts = {
    str(r.book_id): f"{r.title or ''}. {r.description or ''}"
    for r in books_df.itertuples()
}
embed_and_save(book_ids_sorted, book_texts, "book_id",
               OUT_DIR / "books_bge_base_768.h5")

# ── Reviews: review_text ─────────────────────────────────────────────────────
review_texts = {str(r.review_id): r.review_text for r in reviews_df.itertuples()}
embed_and_save(review_ids_sorted, review_texts, "review_id",
               OUT_DIR / "reviews_bge_base_768.h5")

# ── Authors: name ────────────────────────────────────────────────────────────
author_texts = {str(r.author_id): (r.name or "") for r in authors_df.itertuples()}
embed_and_save(author_ids_sorted, author_texts, "author_id",
               OUT_DIR / "authors_bge_base_768.h5",
               fallback="[unknown author]")

# ── Works: original_title with best_book title fallback (via books_df) ──────
book_title_by_id = dict(zip(books_df["book_id"].astype(str), books_df["title"].fillna("")))

def resolve_work_title(row):
    t = (row.original_title or "").strip() if hasattr(row, "original_title") else ""
    if t:
        return t
    if hasattr(row, "best_book_id") and row.best_book_id is not None:
        t = book_title_by_id.get(str(row.best_book_id), "")
    return t.strip() or "[no title]"

work_texts = {str(r.work_id): resolve_work_title(r) for r in works_df.itertuples()}
embed_and_save(work_ids_sorted, work_texts, "work_id",
               OUT_DIR / "works_bge_base_768.h5",
               fallback="[no title]")

# Free GPU memory before numeric block.
del emb_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Numeric blocks

One cell per node type. All stats computed over the slice (Phase 1 shortcut).

In [ ]:
# ── Book numeric (9 dims) ────────────────────────────────────────────────────
b = books_df.copy()
b["num_pages"]          = pd.to_numeric(b["num_pages"], errors="coerce")
b["publication_year"]   = pd.to_numeric(b["publication_year"], errors="coerce")
b["average_rating"]     = pd.to_numeric(b["average_rating"], errors="coerce")
b["ratings_count"]      = pd.to_numeric(b["ratings_count"], errors="coerce").fillna(0).astype(int)
b["text_reviews_count"] = pd.to_numeric(b["text_reviews_count"], errors="coerce").fillna(0).astype(int)
b["is_ebook_flag"]      = b["is_ebook"].apply(
    lambda v: 0 if pd.isna(v) else int(bool(v)) if isinstance(v, bool) else int(str(v).strip().lower() == "true"))

b["is_num_pages_missing"]     = b["num_pages"].isna().astype(int)
b["is_year_missing"]          = b["publication_year"].isna().astype(int)
b["is_avg_rating_meaningful"] = (b["ratings_count"] > 0).astype(int)

num_pages_median = float(b["num_pages"].median())
year_mean = float(b["publication_year"].mean())
year_std  = float(b["publication_year"].std())

book_norm_stats = {
    "num_pages_median": num_pages_median,
    "year_mean": year_mean,
    "year_std":  year_std,
    "_note": "Phase 1: full-slice stats. Tighten to training split in Phase 2.",
}
save_json(book_norm_stats, OUT_DIR / "book_norm_stats.json")

b["num_pages_imp"] = b["num_pages"].fillna(num_pages_median)
b["year_norm"]     = ((b["publication_year"] - year_mean) / year_std).fillna(0.0)
b["avg_rating_clean"] = b["average_rating"].where(b["is_avg_rating_meaningful"] == 1, 0.0).fillna(0.0)

# Index by book_id so rows align with book_ids_sorted (lex order).
b = b.set_index("book_id")
ordered = b.loc[book_ids_sorted]

book_numeric = np.stack([
    np.log1p(ordered["num_pages_imp"].to_numpy(dtype=np.float64)),
    ordered["year_norm"].to_numpy(dtype=np.float64),
    ordered["avg_rating_clean"].to_numpy(dtype=np.float64),
    np.log1p(ordered["ratings_count"].to_numpy(dtype=np.float64)),
    np.log1p(ordered["text_reviews_count"].to_numpy(dtype=np.float64)),
    ordered["is_ebook_flag"].to_numpy(dtype=np.float64),
    ordered["is_num_pages_missing"].to_numpy(dtype=np.float64),
    ordered["is_year_missing"].to_numpy(dtype=np.float64),
    ordered["is_avg_rating_meaningful"].to_numpy(dtype=np.float64),
], axis=1).astype(np.float32)
assert book_numeric.shape == (len(book_ids_sorted), 9), book_numeric.shape

np.save(OUT_DIR / "book_numeric.npy", book_numeric)
save_json([
    "log1p_num_pages", "year_normalized", "average_rating",
    "log1p_ratings_count", "log1p_text_reviews_count",
    "is_ebook", "is_num_pages_missing", "is_year_missing", "is_avg_rating_meaningful",
], OUT_DIR / "book_numeric_columns.json")
print(f"  book_numeric  {book_numeric.shape} → {OUT_DIR / 'book_numeric.npy'}")

In [ ]:
# ── Author numeric (4 dims) ──────────────────────────────────────────────────
a = authors_df.copy().set_index("author_id")
a = a.reindex(author_ids_sorted)

a_ratings_count      = a["ratings_count"].fillna(0).clip(lower=0)
a_text_reviews_count = a["text_reviews_count"].fillna(0).clip(lower=0)
a_is_meaningful      = (a_ratings_count > 0).astype(int)
a_avg_rating         = a["average_rating"].where(a_is_meaningful == 1, 0.0).fillna(0.0)

author_numeric = np.stack([
    a_avg_rating.to_numpy(dtype=np.float64),
    np.log1p(a_ratings_count.to_numpy(dtype=np.float64)),
    np.log1p(a_text_reviews_count.to_numpy(dtype=np.float64)),
    a_is_meaningful.to_numpy(dtype=np.float64),
], axis=1).astype(np.float32)
assert author_numeric.shape == (len(author_ids_sorted), 4), author_numeric.shape

np.save(OUT_DIR / "author_numeric.npy", author_numeric)
save_json([
    "average_rating",
    "log1p_ratings_count",
    "log1p_text_reviews_count",
    "is_avg_rating_meaningful",
], OUT_DIR / "author_numeric_columns.json")
print(f"  author_numeric  {author_numeric.shape} → {OUT_DIR / 'author_numeric.npy'}")

In [ ]:
# ── Work numeric (11 dims) ───────────────────────────────────────────────────
w = works_df.copy().set_index("work_id").reindex(work_ids_sorted)

w_years = w["original_publication_year"]
year_mean_w = float(w_years.dropna().mean())
year_std_w  = float(w_years.dropna().std())
w_year_norm = ((w_years - year_mean_w) / year_std_w).fillna(0.0)
w_is_year_missing = w_years.isna().astype(int)

w_books_count        = w["books_count"].fillna(0).clip(lower=0)
w_ratings_count      = w["ratings_count"].fillna(0).clip(lower=0)
w_text_reviews_count = w["text_reviews_count"].fillna(0).clip(lower=0)

w_rd_total = w["rd_total"].fillna(0).clip(lower=0)
w_is_meaningful = (w_rd_total > 0).astype(int)

def rd_norm(col):
    v = w[col].fillna(0)
    return np.where(w_rd_total > 0, v / w_rd_total.replace(0, np.nan), 0.0)

work_numeric = np.stack([
    w_year_norm.to_numpy(dtype=np.float64),
    np.log1p(w_books_count.to_numpy(dtype=np.float64)),
    np.log1p(w_ratings_count.to_numpy(dtype=np.float64)),
    np.log1p(w_text_reviews_count.to_numpy(dtype=np.float64)),
    rd_norm("rd_5"),
    rd_norm("rd_4"),
    rd_norm("rd_3"),
    rd_norm("rd_2"),
    rd_norm("rd_1"),
    w_is_year_missing.to_numpy(dtype=np.float64),
    w_is_meaningful.to_numpy(dtype=np.float64),
], axis=1).astype(np.float32)
assert work_numeric.shape == (len(work_ids_sorted), 11), work_numeric.shape

np.save(OUT_DIR / "work_numeric.npy", work_numeric)
save_json([
    "original_year_normalized", "log1p_books_count",
    "log1p_ratings_count", "log1p_text_reviews_count",
    "rd_5_norm", "rd_4_norm", "rd_3_norm", "rd_2_norm", "rd_1_norm",
    "is_year_missing", "is_avg_rating_meaningful",
], OUT_DIR / "work_numeric_columns.json")
save_json({
    "year_mean": year_mean_w, "year_std": year_std_w,
    "_note": "Phase 1: full-slice stats. Tighten to training split in Phase 2.",
}, OUT_DIR / "work_norm_stats.json")
print(f"  work_numeric  {work_numeric.shape} → {OUT_DIR / 'work_numeric.npy'}")

In [ ]:
# ── Review numeric (4 dims) ──────────────────────────────────────────────────
r = reviews_df.copy().set_index("review_id").reindex(review_ids_sorted)
r_rating     = r["rating"].fillna(0).clip(lower=0)
r_n_votes    = r["n_votes"].fillna(0).clip(lower=0)
r_n_comments = r["n_comments"].fillna(0).clip(lower=0)
r_is_rated   = (r_rating > 0).astype(int)

review_numeric = np.stack([
    r_rating.to_numpy(dtype=np.float64),
    np.log1p(r_n_votes.to_numpy(dtype=np.float64)),
    np.log1p(r_n_comments.to_numpy(dtype=np.float64)),
    r_is_rated.to_numpy(dtype=np.float64),
], axis=1).astype(np.float32)
assert review_numeric.shape == (len(review_ids_sorted), 4), review_numeric.shape

np.save(OUT_DIR / "review_numeric.npy", review_numeric)
save_json(["rating", "log1p_n_votes", "log1p_n_comments", "is_rated"],
          OUT_DIR / "review_numeric_columns.json")
print(f"  review_numeric  {review_numeric.shape} → {OUT_DIR / 'review_numeric.npy'}")

## 8. Derive edges from Mongo

All edge types in one cell. Each is stored as a NumPy array of integer index
pairs `(src_idx, dst_idx)` (plus weight where applicable) — handed to step 10.

In [ ]:
edges: dict[tuple, dict] = {}   # (src, rel, dst) → {edge_index: ndarray, weight?: ndarray, ...}


def to_idx_pairs(src_ids, dst_ids, src_map, dst_map):
    src = np.array([src_map[s] for s in src_ids], dtype=np.int64)
    dst = np.array([dst_map[d] for d in dst_ids], dtype=np.int64)
    return np.stack([src, dst], axis=0)


# ── User → Book (4 sub-types) with date_added ────────────────────────────────
def assign_edge_type(row):
    if row["is_read"]:
        if row["rating"] >= 4:  return "RATED_HIGH"
        if row["rating"] >= 1:  return "RATED_LOW"
        return "READ_UNRATED"
    return "SHELVED"

inters_df["edge_type"] = inters_df.apply(assign_edge_type, axis=1)

for etype in ["RATED_HIGH", "RATED_LOW", "READ_UNRATED", "SHELVED"]:
    sub = inters_df[inters_df["edge_type"] == etype]
    sub = sub[sub["user_id"].isin(user_to_idx) & sub["book_id"].isin(book_to_idx)]
    if len(sub) == 0:
        edges["user", etype, "book"] = {
            "edge_index": np.empty((2, 0), dtype=np.int64),
            "date_added": np.array([], dtype="datetime64[ns]"),
            "rating":     np.array([], dtype=np.int8),
        }
        continue
    ei = to_idx_pairs(sub["user_id"].tolist(), sub["book_id"].tolist(), user_to_idx, book_to_idx)
    edges["user", etype, "book"] = {
        "edge_index": ei,
        "date_added": sub["date_added"].to_numpy(),
        "rating":     sub["rating"].to_numpy(dtype=np.int8),
    }
    print(f"  (user, {etype:<13}, book)  {ei.shape[1]:>8,}")

# ── User → Review : WROTE ────────────────────────────────────────────────────
sub = reviews_df[reviews_df["user_id"].isin(user_to_idx) & reviews_df["review_id"].isin(review_to_idx)]
ei = to_idx_pairs(sub["user_id"].tolist(), sub["review_id"].tolist(), user_to_idx, review_to_idx)
edges["user", "WROTE", "review"] = {"edge_index": ei}
print(f"  (user, WROTE, review)        {ei.shape[1]:>8,}")

# ── Review → Book : REVIEWS ──────────────────────────────────────────────────
sub = reviews_df[reviews_df["book_id"].isin(book_to_idx)]
ei = to_idx_pairs(sub["review_id"].tolist(), sub["book_id"].tolist(), review_to_idx, book_to_idx)
edges["review", "REVIEWS", "book"] = {"edge_index": ei}
print(f"  (review, REVIEWS, book)      {ei.shape[1]:>8,}")

# ── Book → Author : AUTHORED_BY (with role + position) ───────────────────────
authored_rows = []
for r in books_df.itertuples():
    for i, a in enumerate(r.authors or []):
        if a["author_id"] in author_to_idx:
            authored_rows.append((r.book_id, a["author_id"], a.get("role") or "", i))
if authored_rows:
    src_ids, dst_ids, roles, positions = zip(*authored_rows)
    ei = to_idx_pairs(list(src_ids), list(dst_ids), book_to_idx, author_to_idx)
    edges["book", "AUTHORED_BY", "author"] = {
        "edge_index": ei,
        "position":   np.array(positions, dtype=np.int16),
    }
    print(f"  (book, AUTHORED_BY, author)  {ei.shape[1]:>8,}")

# ── Book → Work : EDITION_OF ─────────────────────────────────────────────────
sub = books_df.dropna(subset=["work_id"])
sub = sub[sub["work_id"].astype(str).isin(work_to_idx)]
ei  = to_idx_pairs(sub["book_id"].tolist(), sub["work_id"].astype(str).tolist(), book_to_idx, work_to_idx)
edges["book", "EDITION_OF", "work"] = {"edge_index": ei}
print(f"  (book, EDITION_OF, work)     {ei.shape[1]:>8,}")

# ── Book → Genre : HAS_GENRE (normalized weights, optional min-weight filter) ─
def norm_dict(d):
    if not d: return {}
    s = sum(d.values())
    return {k: v / s for k, v in d.items()} if s else {}

gen_lookup = {row["book_id"]: norm_dict(row["genres"]) for _, row in genres_df.iterrows()}
gen_rows = []
n_dropped_below_threshold = 0
n_dropped_unknown_genre   = 0
for bid in books_df["book_id"]:
    for g, w in (gen_lookup.get(bid) or {}).items():
        if g not in genre_to_idx:
            n_dropped_unknown_genre += 1
            continue
        if w < GENRE_MIN_WEIGHT:
            n_dropped_below_threshold += 1
            continue
        gen_rows.append((bid, g, float(w)))
if gen_rows:
    src_ids, dst_ids, ws = zip(*gen_rows)
    ei = to_idx_pairs(list(src_ids), list(dst_ids), book_to_idx, genre_to_idx)
    edges["book", "HAS_GENRE", "genre"] = {
        "edge_index": ei,
        "weight":     np.array(ws, dtype=np.float32),
    }
    n_books_with_any_genre = pd.DataFrame(gen_rows, columns=["bid","g","w"])["bid"].nunique()
    avg_per_book = ei.shape[1] / max(n_books_with_any_genre, 1)
    print(f"  (book, HAS_GENRE, genre)     {ei.shape[1]:>8,}  "
          f"(min_weight={GENRE_MIN_WEIGHT}, dropped {n_dropped_below_threshold:,} below threshold; "
          f"avg {avg_per_book:.2f}/book over {n_books_with_any_genre:,} books with ≥1 kept genre)")

# ── Book → Shelf : HAS_SHELF (top-N, weights from per-book counts) ───────────
KEEP = set(shelf_to_idx)
shelf_rows = []
for r in books_df.itertuples():
    if not r.popular_shelves:
        continue
    kept = [(s["name"], int(s["count"])) for s in r.popular_shelves if s["name"] in KEEP]
    total = sum(c for _, c in kept)
    if total == 0:
        continue
    for name, c in kept:
        shelf_rows.append((r.book_id, name, c / total))
if shelf_rows:
    src_ids, dst_ids, ws = zip(*shelf_rows)
    ei = to_idx_pairs(list(src_ids), list(dst_ids), book_to_idx, shelf_to_idx)
    edges["book", "HAS_SHELF", "shelf"] = {
        "edge_index": ei,
        "weight":     np.array(ws, dtype=np.float32),
    }
    print(f"  (book, HAS_SHELF, shelf)     {ei.shape[1]:>8,}")

# ── Book → Language / Format / Publisher (one row per book) ──────────────────
for col, rel, dst_map, dst_name in [
    ("language_node",  "IN_LANGUAGE",  lang_to_idx, "language"),
    ("format_node",    "IN_FORMAT",    fmt_to_idx,  "format"),
    ("publisher_node", "PUBLISHED_BY", pub_to_idx,  "publisher"),
]:
    sub = books_df[["book_id", col]].dropna()
    sub = sub[sub[col].isin(dst_map)]
    ei  = to_idx_pairs(sub["book_id"].tolist(), sub[col].tolist(), book_to_idx, dst_map)
    edges["book", rel, dst_name] = {"edge_index": ei}
    print(f"  (book, {rel:<12}, {dst_name:<9})  {ei.shape[1]:>8,}")

## 9. Temporal splits

Train < T-2yr, val [T-2yr, T-1yr), test [T-1yr, T]. Null `date_added` → train.

**Supervision label policy:**
- `RATED_HIGH` (rating ≥ 4) → label = 1 (positive)
- `RATED_LOW`  (rating 1–3) → label = 0 (explicit negative)
- `READ_UNRATED` and `SHELVED` → **not in supervision**. Both remain as graph-only edges in step 10 (READ_UNRATED filtered to pre-T_val so val/test rows don't leak into the message-passing graph).

Rationale: "engaged with" ≠ "preferred". Treating READ_UNRATED as positive injects ~10% noisy labels into training. Keeping it as a graph edge still lets the model use the "user read this" signal for representation learning without committing to a prediction target.

In [ ]:
EDGE_TYPE_CODE = {"RATED_HIGH": 0, "RATED_LOW": 1}
LABEL          = {"RATED_HIGH": 1, "RATED_LOW": 0}

sup_chunks = []
for etype in ["RATED_HIGH", "RATED_LOW"]:
    e = edges["user", etype, "book"]
    if e["edge_index"].shape[1] == 0:
        continue
    sup_chunks.append(pd.DataFrame({
        "user_idx":  e["edge_index"][0],
        "book_idx":  e["edge_index"][1],
        "edge_type": np.full(e["edge_index"].shape[1], EDGE_TYPE_CODE[etype], dtype=np.int8),
        "label":     np.full(e["edge_index"].shape[1], LABEL[etype],          dtype=np.int8),
        "date_added": pd.Series(e["date_added"]),
    }))

sup_df = pd.concat(sup_chunks, ignore_index=True)
sup_df["date_added"] = pd.to_datetime(sup_df["date_added"], utc=True, errors="coerce")

T = sup_df["date_added"].max()
T_val  = T - pd.DateOffset(years=TRAIN_YEARS_BEFORE_T)
T_test = T - pd.DateOffset(years=VAL_YEARS_BEFORE_T)
print(f"  T       = {T.date()}")
print(f"  T_val   = {T_val.date()}  (train/val boundary)")
print(f"  T_test  = {T_test.date()}  (val/test boundary)")

is_val  = sup_df["date_added"].between(T_val, T_test, inclusive="left")
is_test = sup_df["date_added"] >= T_test
sup_df["split"] = np.where(is_test, "test", np.where(is_val, "val", "train"))

stats = {"T": str(T.date()), "T_val": str(T_val.date()), "T_test": str(T_test.date()), "splits": {}}
for split in ["train", "val", "test"]:
    sub = sup_df[sup_df["split"] == split]
    pos = int((sub["label"] == 1).sum())
    neg = int((sub["label"] == 0).sum())
    path = SPLITS_DIR / f"{split}.npz"
    np.savez(
        path,
        user_idx=sub["user_idx"].to_numpy(dtype=np.int32),
        book_idx=sub["book_idx"].to_numpy(dtype=np.int32),
        edge_type=sub["edge_type"].to_numpy(dtype=np.int8),
        label=sub["label"].to_numpy(dtype=np.int8),
    )
    stats["splits"][split] = {"total": len(sub), "pos": pos, "neg": neg}
    print(f"  {split:<6}  total={len(sub):>8,}  pos={pos:>8,}  neg={neg:>8,}  → {path}")

save_json(stats, SPLITS_DIR / "split_stats.json")

## 10. Assemble `HeteroData`

Pulls every artifact written above into a single PyG `HeteroData` object.

**Edge inclusion rules:**

| Edge type | Source | Time filter |
|---|---|---|
| Structural (`AUTHORED_BY`, `EDITION_OF`, `HAS_GENRE`, `HAS_SHELF`, `PUBLISHED_BY`, `IN_LANGUAGE`, `IN_FORMAT`, `WROTE`, `REVIEWS`) | `edges` dict (full) | none |
| `SHELVED` | `edges` dict (full) | none — no rating, low leakage risk |
| `READ_UNRATED` | `edges` dict | **date_added < T_val** (graph-only, no leak into val/test) |
| `RATED_HIGH` / `RATED_LOW` (supervision) | `splits/train.npz` | train portion only (date_added < T_val by construction) |

Reverse edges added via `T.ToUndirected(merge=False)` — HGT treats each direction as a distinct relation.

In [ ]:
from torch_geometric.data import HeteroData
import torch_geometric.transforms as PygT

data = HeteroData()

# ── U-category: num_nodes only (ID embeddings live in the model) ─────────────
for nt, m in [("user", user_to_idx), ("genre", genre_to_idx), ("shelf", shelf_to_idx),
              ("language", lang_to_idx), ("format", fmt_to_idx), ("publisher", pub_to_idx)]:
    data[nt].num_nodes = len(m)

# ── TN/MN: x = concat(text_embed, numeric) ──────────────────────────────────
def load_x(h5_name: str, npy_name: str) -> torch.Tensor:
    with h5py.File(OUT_DIR / h5_name, "r") as hf:
        emb = hf["embedding"][:]
    num = np.load(OUT_DIR / npy_name)
    assert emb.shape[0] == num.shape[0], (emb.shape, num.shape)
    return torch.from_numpy(np.concatenate([emb, num], axis=1)).float()

data["book"].x   = load_x("books_bge_base_768.h5",   "book_numeric.npy")
data["review"].x = load_x("reviews_bge_base_768.h5", "review_numeric.npy")
data["author"].x = load_x("authors_bge_base_768.h5", "author_numeric.npy")
data["work"].x   = load_x("works_bge_base_768.h5",   "work_numeric.npy")

for nt in ["book", "review", "author", "work"]:
    print(f"  {nt:<7} x={tuple(data[nt].x.shape)}")

# ── Per-node timestamps (years since 1900) for the RTE ablation ──────────────
# These are always written so cfg.use_rte can be toggled without rerunning preprocessing.
print("\n[+] Computing node timestamps (years since 1900) for RTE ablation...")

def _years_since_1900(s):
    """date Series → float years (year + day_of_year/365), NaT → NaN."""
    s = pd.to_datetime(s, utc=True, errors="coerce")
    return (s.dt.year + s.dt.dayofyear / 365.0) - 1900.0

# Book: publication_year directly (already numeric)
book_years = pd.to_numeric(
    books_df.set_index("book_id").reindex(book_ids_sorted)["publication_year"],
    errors="coerce",
)
book_median = float(book_years.median())
book_t = (book_years.fillna(book_median) - 1900.0).to_numpy(dtype=np.float32)
data["book"].t = torch.from_numpy(book_t)

# Review: date_added
rev_dates = _years_since_1900(reviews_df.set_index("review_id")
                              .reindex(review_ids_sorted)["date_added"])
rev_t = rev_dates.fillna(rev_dates.median()).to_numpy(dtype=np.float32)
data["review"].t = torch.from_numpy(rev_t)

# User: latest interaction across (interactions ∪ reviews)
_user_dates = pd.concat([
    inters_df[["user_id", "date_added"]],
    reviews_df[["user_id", "date_added"]],
], ignore_index=True)
_user_dates["t"] = _years_since_1900(_user_dates["date_added"])
user_max_t = (_user_dates.groupby("user_id")["t"].max()
              .reindex(user_ids_sorted))
user_t = user_max_t.fillna(user_max_t.median()).to_numpy(dtype=np.float32)
data["user"].t = torch.from_numpy(user_t)

# Work: average pub_year of its books (via EDITION_OF)
ei_bw = edges["book", "EDITION_OF", "work"]["edge_index"]
work_t_sum = np.zeros(len(work_to_idx), dtype=np.float64)
work_t_cnt = np.zeros(len(work_to_idx), dtype=np.float64)
np.add.at(work_t_sum, ei_bw[1], book_t[ei_bw[0]])
np.add.at(work_t_cnt, ei_bw[1], 1.0)
work_median = float(np.median(book_t))
work_t = np.where(work_t_cnt > 0, work_t_sum / np.maximum(work_t_cnt, 1), work_median).astype(np.float32)
data["work"].t = torch.from_numpy(work_t)

# Author: average pub_year of their books (via AUTHORED_BY)
ei_ba = edges["book", "AUTHORED_BY", "author"]["edge_index"]
auth_t_sum = np.zeros(len(author_to_idx), dtype=np.float64)
auth_t_cnt = np.zeros(len(author_to_idx), dtype=np.float64)
np.add.at(auth_t_sum, ei_ba[1], book_t[ei_ba[0]])
np.add.at(auth_t_cnt, ei_ba[1], 1.0)
auth_t = np.where(auth_t_cnt > 0, auth_t_sum / np.maximum(auth_t_cnt, 1), work_median).astype(np.float32)
data["author"].t = torch.from_numpy(auth_t)

# U-category (Genre / Shelf / Language / Format / Publisher): constant slice median.
# These types don't have a natural timestamp; RTE contribution is uniform across them.
for nt in ["genre", "shelf", "language", "format", "publisher"]:
    data[nt].t = torch.full((data[nt].num_nodes,), float(work_median), dtype=torch.float32)

print(f"  book.t     mean={book_t.mean():.2f}   review.t   mean={rev_t.mean():.2f}")
print(f"  user.t     mean={user_t.mean():.2f}   work.t     mean={work_t.mean():.2f}   author.t mean={auth_t.mean():.2f}")
print(f"  u-category mean={work_median:.2f}  (constant)")

# ── Structural edges (all non-supervision, non-time-filtered relations) ─────
STRUCTURAL = [
    ("book",   "AUTHORED_BY",  "author"),
    ("book",   "EDITION_OF",   "work"),
    ("book",   "HAS_GENRE",    "genre"),
    ("book",   "HAS_SHELF",    "shelf"),
    ("book",   "PUBLISHED_BY", "publisher"),
    ("book",   "IN_LANGUAGE",  "language"),
    ("book",   "IN_FORMAT",    "format"),
    ("user",   "WROTE",        "review"),
    ("review", "REVIEWS",      "book"),
    ("user",   "SHELVED",      "book"),
]
for key in STRUCTURAL:
    e = edges.get(key)
    if e is None:
        continue
    ei = torch.from_numpy(e["edge_index"]).long()
    data[key[0], key[1], key[2]].edge_index = ei
    if "weight" in e:
        data[key[0], key[1], key[2]].edge_weight = torch.from_numpy(e["weight"]).float()
    print(f"  {key} edges={ei.shape[1]:,}")

# ── READ_UNRATED: graph-only, filtered to pre-T_val to avoid temporal leak ──
split_stats = json.loads((SPLITS_DIR / "split_stats.json").read_text(encoding="utf-8"))
T_val_dt = pd.Timestamp(split_stats["T_val"], tz="UTC")
ru = edges["user", "READ_UNRATED", "book"]
ru_dates = pd.to_datetime(pd.Series(ru["date_added"]), utc=True, errors="coerce")
ru_mask = (ru_dates < T_val_dt).to_numpy()
ru_ei = torch.from_numpy(ru["edge_index"][:, ru_mask]).long()
data["user", "READ_UNRATED", "book"].edge_index = ru_ei
print(f"  (user, READ_UNRATED, book) graph-only edges={ru_ei.shape[1]:,}  "
      f"(dropped {(~ru_mask).sum():,} post-T_val rows)")

# ── Supervision edges: TRAIN portion only goes into the graph ────────────────
train_z = np.load(SPLITS_DIR / "train.npz")
for code_int, rel in [(0, "RATED_HIGH"), (1, "RATED_LOW")]:
    m = train_z["edge_type"] == code_int
    ei = torch.from_numpy(np.stack([train_z["user_idx"][m], train_z["book_idx"][m]])).long()
    data["user", rel, "book"].edge_index = ei
    print(f"  (user, {rel}, book) train-only edges={ei.shape[1]:,}")

# ── Reverse edges ────────────────────────────────────────────────────────────
data = PygT.ToUndirected(merge=False)(data)
print(f"  total edge types after ToUndirected: {len(data.edge_types)}")

data.validate(raise_on_error=True)
torch.save(data, OUT_DIR / "hetero_data.pt")

meta = {
    "node_types":        {nt: int(data[nt].num_nodes) for nt in data.node_types},
    "node_feature_dims": {nt: (int(data[nt].x.shape[1]) if "x" in data[nt] else 0)
                          for nt in data.node_types},
    "node_has_t":        {nt: ("t" in data[nt]) for nt in data.node_types},
    "edge_counts": {f"{s}__{r}__{d}": int(data[s, r, d].edge_index.shape[1])
                    for (s, r, d) in data.edge_types},
}
save_json(meta, OUT_DIR / "hetero_meta.json")
print(f"\nSaved {OUT_DIR / 'hetero_data.pt'}  "
      f"({(OUT_DIR / 'hetero_data.pt').stat().st_size / 1024**2:.1f} MB)")

## 11. Sanity check

In-place verification of everything written above. No separate notebook.

In [ ]:
print("== Sanity check ==")

# HDF5 vs idx-map alignment
for h5_name, id_key, idx_map in [
    ("books_bge_base_768.h5",   "book_id",   book_to_idx),
    ("reviews_bge_base_768.h5", "review_id", review_to_idx),
    ("authors_bge_base_768.h5", "author_id", author_to_idx),
    ("works_bge_base_768.h5",   "work_id",   work_to_idx),
]:
    with h5py.File(OUT_DIR / h5_name, "r") as hf:
        ids = [x.decode() for x in hf[id_key][:]]
        emb = hf["embedding"]
        assert emb.shape == (len(idx_map), EMB_DIM), (h5_name, emb.shape)
        assert emb.dtype == np.float32, (h5_name, emb.dtype)
    assert ids == sorted(ids), f"{h5_name} not lex-sorted"
    assert idx_map[ids[0]] == 0 and idx_map[ids[-1]] == len(ids) - 1, h5_name
    print(f"  {h5_name:<28}  shape={emb.shape}  row0='{ids[0]}'  rowN='{ids[-1]}'")

# Numeric blocks shape
for npy_name, expected_dim in [
    ("book_numeric.npy",   9),
    ("author_numeric.npy", 4),
    ("work_numeric.npy",  11),
    ("review_numeric.npy", 4),
]:
    arr = np.load(OUT_DIR / npy_name)
    assert arr.dtype == np.float32, (npy_name, arr.dtype)
    assert arr.shape[1] == expected_dim, (npy_name, arr.shape)
    print(f"  {npy_name:<22}  shape={arr.shape}")

# Edge index bounds
n_per_type = {**{k: data[k].num_nodes for k in data.node_types}}
for (s, r, d) in data.edge_types:
    ei = data[s, r, d].edge_index
    if ei.numel() == 0:
        continue
    assert ei[0].max().item() < n_per_type[s], (s, r, d)
    assert ei[1].max().item() < n_per_type[d], (s, r, d)
print("  all edge_index in-bounds")

# Split totals — RATED_HIGH + RATED_LOW only (READ_UNRATED is graph-only)
total_sup = 0
for split in ["train", "val", "test"]:
    z = np.load(SPLITS_DIR / f"{split}.npz")
    assert z["user_idx"].shape == z["book_idx"].shape == z["label"].shape
    total_sup += z["user_idx"].shape[0]
n_label_edges = sum(edges["user", e, "book"]["edge_index"].shape[1]
                    for e in ["RATED_HIGH", "RATED_LOW"])
assert total_sup == n_label_edges, (total_sup, n_label_edges)
print(f"  splits sum ({total_sup:,}) == labeled supervision ({n_label_edges:,})  "
      f"[READ_UNRATED graph-only: {edges['user', 'READ_UNRATED', 'book']['edge_index'].shape[1]:,}]")

# READ_UNRATED graph edge present in HeteroData, train-period only
ru_in_graph = data["user", "READ_UNRATED", "book"].edge_index.shape[1]
ru_total    = edges["user", "READ_UNRATED", "book"]["edge_index"].shape[1]
assert ru_in_graph <= ru_total, (ru_in_graph, ru_total)
print(f"  READ_UNRATED in graph: {ru_in_graph:,}/{ru_total:,}  "
      f"({100*ru_in_graph/max(ru_total,1):.1f}% — rest is post-T_val, excluded)")

print("\n✅ All checks passed.")

## 12. (Optional) Neo4j export

Runs only when `PUSH_TO_NEO4J = True`. Writes nodes + edges into Neo4j DB
`{SLICE_NAME}` for visualization. Never read back by training. Skip if you
don't have Neo4j running or just want the artifacts.

In [ ]:
if not PUSH_TO_NEO4J:
    print("Neo4j export disabled (set PUSH_TO_NEO4J = True in the config cell).")
else:
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

    def with_session(fn):
        with driver.session(database=NEO4J_DB) as s:
            return fn(s)

    # --- Indexes ---
    def create_indexes(tx):
        for lbl, prop in [
            ("Book", "book_id"), ("Author", "author_id"), ("Work", "work_id"),
            ("User", "user_id"), ("Review", "review_id"),
            ("Genre", "name"), ("Shelf", "name"),
            ("Language", "code"), ("Format", "name"), ("Publisher", "name"),
        ]:
            tx.run(f"CREATE INDEX {lbl.lower()}_{prop}_idx IF NOT EXISTS "
                   f"FOR (n:{lbl}) ON (n.{prop})")
    with_session(lambda s: s.execute_write(create_indexes))

    # --- Helper: bulk node MERGE ---
    def push_nodes(label: str, key_prop: str, items: list):
        def tx(t, batch):
            t.run(f"UNWIND $batch AS x MERGE (:{label} {{{key_prop}: x[$k]}})",
                  batch=batch, k=key_prop)
        for chunk in chunked([{key_prop: v} for v in items], 10_000):
            with_session(lambda s: s.execute_write(tx, chunk))
        print(f"  pushed {len(items):,} {label}")

    push_nodes("Book",      "book_id",   book_ids_sorted)
    push_nodes("Author",    "author_id", author_ids_sorted)
    push_nodes("Work",      "work_id",   work_ids_sorted)
    push_nodes("User",      "user_id",   user_ids_sorted)
    push_nodes("Review",    "review_id", review_ids_sorted)
    push_nodes("Genre",     "name",      list(genre_to_idx))
    push_nodes("Shelf",     "name",      list(shelf_to_idx))
    push_nodes("Language",  "code",      list(lang_to_idx))
    push_nodes("Format",    "name",      list(fmt_to_idx))
    push_nodes("Publisher", "name",      list(pub_to_idx))

    # --- Helper: bulk edge MERGE ---
    EDGE_PUSH = {
        ("book", "AUTHORED_BY", "author"):  ("Book",   "book_id",   "Author",    "author_id"),
        ("book", "EDITION_OF",  "work"):    ("Book",   "book_id",   "Work",      "work_id"),
        ("book", "HAS_GENRE",   "genre"):   ("Book",   "book_id",   "Genre",     "name"),
        ("book", "HAS_SHELF",   "shelf"):   ("Book",   "book_id",   "Shelf",     "name"),
        ("book", "IN_LANGUAGE", "language"):("Book",   "book_id",   "Language",  "code"),
        ("book", "IN_FORMAT",   "format"):  ("Book",   "book_id",   "Format",    "name"),
        ("book", "PUBLISHED_BY","publisher"):("Book",  "book_id",   "Publisher", "name"),
        ("user", "WROTE",       "review"):  ("User",   "user_id",   "Review",    "review_id"),
        ("review","REVIEWS",    "book"):    ("Review", "review_id", "Book",      "book_id"),
        ("user", "SHELVED",     "book"):    ("User",   "user_id",   "Book",      "book_id"),
        ("user", "RATED_HIGH",  "book"):    ("User",   "user_id",   "Book",      "book_id"),
        ("user", "RATED_LOW",   "book"):    ("User",   "user_id",   "Book",      "book_id"),
        ("user", "READ_UNRATED","book"):    ("User",   "user_id",   "Book",      "book_id"),
    }
    # Reverse maps to recover string IDs from integer indices
    inv_user   = {v: k for k, v in user_to_idx.items()}
    inv_book   = {v: k for k, v in book_to_idx.items()}
    inv_author = {v: k for k, v in author_to_idx.items()}
    inv_work   = {v: k for k, v in work_to_idx.items()}
    inv_review = {v: k for k, v in review_to_idx.items()}
    inv_genre  = {v: k for k, v in genre_to_idx.items()}
    inv_shelf  = {v: k for k, v in shelf_to_idx.items()}
    inv_lang   = {v: k for k, v in lang_to_idx.items()}
    inv_fmt    = {v: k for k, v in fmt_to_idx.items()}
    inv_pub    = {v: k for k, v in pub_to_idx.items()}
    INV = {"user": inv_user, "book": inv_book, "author": inv_author, "work": inv_work,
           "review": inv_review, "genre": inv_genre, "shelf": inv_shelf,
           "language": inv_lang, "format": inv_fmt, "publisher": inv_pub}

    for (src_t, rel, dst_t), (S_lbl, S_key, D_lbl, D_key) in EDGE_PUSH.items():
        e = edges.get((src_t, rel, dst_t))
        if e is None or e["edge_index"].shape[1] == 0:
            continue
        ei = e["edge_index"]
        items = [{"s": INV[src_t][int(s)], "d": INV[dst_t][int(d)]}
                 for s, d in zip(ei[0], ei[1])]
        def tx(t, batch):
            t.run(f"""
                UNWIND $batch AS e
                MATCH (s:{S_lbl} {{{S_key}: e.s}})
                MATCH (d:{D_lbl} {{{D_key}: e.d}})
                MERGE (s)-[:{rel}]->(d)
            """, batch=batch)
        for chunk in chunked(items, 10_000):
            with_session(lambda s: s.execute_write(tx, chunk))
        print(f"  pushed {len(items):,} {rel}")

    driver.close()
    print("Neo4j export complete.")

---
*End of pipeline.*